# **Unidad 5. Evaluación de traducciones automáticas**

Este cuaderno es una introducción para aprender cómo se puede evaluar una traducción automática.

En esta unidad veremos:
1. Métricas léxicas básicas: precision, recall, F1-score, índice Jaccard y exact match
2. Métricas automáticas basadas en coincidencia formal: BLEU, chrF y (en un segundo cuaderno:) METEOR

🧠 No necesitas saber programación avanzada.

## 📌 1. ¿Qué es una traducción automática?

Es aquella en la que un sistema (como Google Translate) traduce un texto de un idioma a otro sin intervención humana.

🔄 Traduce, por ejemplo, del catalán al español o del inglés al francés.

## 🤔 2. ¿Por qué habría que evaluarla?

No todas las traducciones son buenas. Queremos saber si el sistema lo ha hecho bien o mal.

**Ejemplo:**
- Original: *Els arbres són alts.*
- Traducción 1: *Los árboles son altos.* ✅
- Traducción 2: *Los arboles estan grandes.* ❌

## 3. Métricas léxicas básicas

### 3.1. Precision (precisión léxica)

* ¿Qué mide? Qué proporción de palabras de la traducción automática coinciden con las de la traducción humana.

* ✔️ Muy fácil
* ❌ No distingue orden ni sinónimos

#### **Precision con traducciones manuales**

Contamos cuántas palabras de la traducción automática coinciden con una traducción de referencia (hecha por un humano).

In [ ]:
def limpiar_palabra(palabra):
    return palabra.strip('.,;:!?"¡¿()[]{}')

def procesar_texto(texto):
    texto = texto.lower()
    palabras = texto.split()
    palabras = [limpiar_palabra(p) for p in palabras]
    return palabras

In [ ]:
referencia = "Los árboles son altos."

traduccion_prueba = "Los arboles estan grandes"

referencia_proc = procesar_texto(referencia) # Frase de referencia procesada
traduccion_prueba_proc = procesar_texto(traduccion_prueba) # Traducción automática de prueba procesada

In [ ]:
print("Referencia:", referencia_proc)
print("Traducción:", traduccion_prueba_proc)

In [ ]:
palabras_correctas = sum([1 for palabra in traduccion_prueba_proc if palabra in referencia_proc])
precision = palabras_correctas / len(traduccion_prueba_proc)

print(f"Precisión: {precision:.2f}")

#### **Precision con traducciones automáticas (Google Translate)**

##### Texto original

In [ ]:
original = "Those trees were big."

##### Traducción humana de referencia

In [ ]:
tradu_humana = "Esos árboles eran enormes."

##### Traducción automática (GoogleTranslator)

In [ ]:
%pip install deep-translator

In [ ]:
from deep_translator import GoogleTranslator
tradu_auto = GoogleTranslator(source='en', target='es').translate(original)

print("Traducción automática:", tradu_auto)


##### Procesado de los textos

In [ ]:
original_proc = procesar_texto(original)
tradu_humana_proc = procesar_texto(tradu_humana)
tradu_auto_proc = procesar_texto(tradu_auto)


In [ ]:
print("Texto original en inglés:", original_proc)
print("Traducción humana de referencia:", tradu_humana_proc)
print("Traducción automática con Google Translate:", tradu_auto_proc)

##### Comparación Google Translate vs. Humana con Precision

In [ ]:
precision = sum(1 for palabra in tradu_auto_proc if palabra in tradu_humana_proc) / len(tradu_auto_proc)

print(f"Precisión: {precision:.2f}")

### 3.2. Recall (cobertura léxica)


* ¿Qué mide? Qué proporción de palabras de la traducción humana fueron recuperadas por la automática.

* ✔️ Fácil de entender
* ✅ Complementa a la precisión
* 🔁 Puedes usar ambas para calcular la F1 si quieres

In [ ]:
recall = sum(1 for palabra in tradu_humana_proc if palabra in tradu_auto_proc) / len(tradu_humana_proc)

In [ ]:
print(f"Recall léxico: {recall:.2f}")

### 3.3. F1-score (media armónica léxica)

* ¿Qué mide? Una combinación equilibrada entre precisión y recall.

* ✔️ Simple
* ❌ Sigue sin captar significado ni sinónimos

In [ ]:
if precision + recall > 0:
    f1 = 2 * (precision * recall) / (precision + recall)
else:
    f1 = 0

In [ ]:
print(f"F1-score: {f1:.2f}")

### 3.4. Exact Match (coincidencias léxicas exactas)

* ¿Qué mide? ¿La traducción automática es exactamente igual a la humana?

* ✔️ Muy simple
* ❌ Solo útil en evaluaciones exactas, sin sinónimos ni variaciones

In [ ]:
exact_match = tradu_auto_proc == tradu_humana_proc

In [ ]:
print(f"Exact Match: {'Sí' if exact_match else 'No'}")

### 3.5. Índice Jaccard (similaridad léxica de conjuntos)


* ¿Qué mide? Cuán similares son los conjuntos de palabras (sin importar orden ni repeticiones).

* ✔️ Muy intuitiva
* ❌ No considera orden

In [ ]:
set_auto = set(tradu_auto_proc)
set_humana = set(tradu_humana_proc)
jaccard = len(set_auto & set_humana) / len(set_auto | set_humana)

In [ ]:
print(f"Índice Jaccard: {jaccard:.2f}")

### 📋 Conjunto de resultados - métricas léxicas

In [ ]:
print(f"Precisión léxica: {precision:.2f}")
print(f"Recall léxico: {recall:.2f}")
print(f"F1-score: {f1:.2f}")
print(f"Exact Match: {'Sí' if exact_match else 'No'}")
print(f"Índice Jaccard: {jaccard:.2f}")

### 📊 Comparación de métricas de léxicas clásicas


| Métrica        | ¿Qué mide?                                                                 | Fórmula / lógica base                                     | Valor típico | Comentario pedagógico                          |
|----------------|-----------------------------------------------------------------------------|------------------------------------------------------------|--------------|------------------------------------------------|
| **Precisión**  | Proporción de palabras de la traducción automática que son correctas       | `correctas / total_automatica`                            | 0–1          | Evalúa cuántas "acierta" la máquina            |
| **Recall**     | Proporción de palabras de la referencia que recupera la automática         | `correctas / total_referencia`                            | 0–1          | Evalúa cuántas "recupera"                      |
| **F1-score**   | Media armónica entre precisión y recall                                    | `2 * (P * R) / (P + R)`                                   | 0–1          | Equilibrio entre precisión y recall            |
| **Exact Match**| Si la traducción automática es *idéntica* a la humana                      | `automatica == referencia`                                | 0 o 1        | Muy estricta; solo útil en tareas cerradas     |
| **Jaccard**    | Similitud entre los conjuntos de palabras (sin importar orden)             | `intersección / unión`                                    | 0–1          | Mide solapamiento léxico sin orden             |


## 4. Métricas automáticas basadas en coincidencia formal

### 4.1. BLEU

BLEU es una métrica más profesional usada en investigación. Vamos a calcularla con `nltk`.

#### 🧠 ¿Qué es BLEU?

BLEU (Bilingual Evaluation Understudy) es una métrica automática que evalúa qué tan buena es una traducción automática comparada con una traducción de referencia (humana).

BLEU funciona contando n-gramas (grupos de 1, 2, 3 o más palabras) que aparecen tanto en la traducción automática como en la humana, y calcula una puntuación de 0 a 1 (o 0% a 100%).

* 1.0 (100%) = traducción perfecta

* 0.0 (0%) = nada coincide

Además, penaliza si la traducción es mucho más corta que la referencia (brevity penalty).

#### Evaluemos la traducción automática con la de referencia con BLEU

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [ ]:
reference = [tradu_humana_proc] 
'''- BLEU espera que las traducciones de referencia estén dentro de una lista de listas porque puedes comparar con más de una referencia.
    - tradu_humana_proc es la traducción humana ya procesada (una lista de palabras).
    - Por eso se pone entre corchetes extra: [tradu_humana_proc]
'''

candidate = tradu_auto_proc 

score = sentence_bleu(reference, candidate, smoothing_function=SmoothingFunction().method1)
'''- sentence_bleu(...) es la función de NLTK para calcular BLEU en frases cortas.
- SmoothingFunction().method1 es importante: ayuda a evitar que BLEU sea cero si no hay coincidencias de bigramas o trigramas.
- En frases cortas, BLEU puede castigar mucho si no hay coincidencias exactas, así que este suavizado mejora el resultado.
'''

print(f"Puntuación BLEU: {score:.2f}")

In [ ]:
print(f"Precisión: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")
print(f"Jaccard: {jaccard:.2f}")
print(f"BLEU: {bleu:.2f}")
print(f"Exact Match: {'Sí' if exact_match else 'No'}")

#### Juguemos con ChatGPT > ¡Pregúntale!


Al haber evaluado las dos frases (el contenido de `tradu_humana_proc` y de `tradu_auto_proc`) con un código determinado para obtener la puntuación BLEU, el resultado nos ha sorprendido. ¿Nos lo podría explicar ChatGPT? 

Es muy importante el prompt. Asegúrate de darle con claridad y de forma muy directa toda la información necesaria para que dé una respuesta acertada.

### 4.2 chrF

#### 🧠 ¿Qué es chrF?

Hasta ahora hemos comparado palabras completas. chrF hace algo parecido, pero con fragmentos de caracteres. Por eso puede reconocer similitudes parciales entre formas como “traducido”, “traducción”, “traducir” o entre palabras con pequeñas variaciones morfológicas. No entiende el significado como lo haría una persona, pero es menos rígida que una coincidencia exacta de palabras.

chrF compara la traducción automática y la traducción humana mediante fragmentos de caracteres.

Así, la métrica puede detectar similitudes parciales entre palabras, incluso cuando no coinciden exactamente.

| Valor de chrF | Lectura orientativa                                                               |
| ------------: | --------------------------------------------------------------------------------- |
|          Bajo | Hay poca similitud formal entre la traducción automática y la referencia.         |
|         Medio | Hay coincidencias parciales, aunque existen diferencias léxicas o estructurales.  |
|          Alto | La traducción automática se parece mucho a la referencia en su forma lingüística. |


Ojo: No comprende plenamente el significado, la adecuación pragmática ni la calidad estilística, pero ayuda a complementar BLEU cuando hay variaciones morfológicas o léxicas.

#### 🧠 ¿Qué es chrF++?

chrF++ incorpora también n-gramas de palabras. Es decir, no solo mira fragmentos de caracteres, sino también secuencias de palabras.

Esto puede ser útil cuando quieres que la métrica tenga algo más en cuenta el orden léxico y la estructura de la frase. Sin embargo, para una unidad introductoria puede resultar menos transparente, porque mezcla dos niveles:

* similitud entre caracteres;
* similitud entre palabras.

En esta unidad y las prácticas y actividades siguientes utilizaremos chrF, porque permite comparar traducciones a partir de n-gramas de caracteres y complementa bien a BLEU. BLEU se basa en coincidencias de palabras y secuencias de palabras, mientras que chrF puede captar semejanzas parciales entre formas lingüísticas. La variante chrF++, que incorpora también n-gramas de palabras, la dejaremos únicamente como ampliación, pero no la incorporaremos a los ejercicios de evaluación.

#### Evaluemos la traducción automática con la de referencia con chrF

In [ ]:
!pip install evaluate sacrebleu -q

In [ ]:
from evaluate import load

chrf = load("chrf")
chrf = chrf.compute(predictions=[tradu_auto], references=[tradu_humana])

chrf

In [ ]:
# Mostramos únicamente la puntuación principal de chrF.
# resultado_chrf es un diccionario, y la clave 'score' contiene el valor de la métrica.
# Usamos :.2f para redondear el resultado a dos decimales y facilitar su lectura.

print(f"chrF: {chrf['score']:.2f}")

In [ ]:
print(f"Precisión: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")
print(f"Jaccard: {jaccard:.2f}")
print(f"BLEU: {bleu:.2f}")
print(f"chrF: {chrf['score']:.2f}")
print(f"Exact Match: {'Sí' if exact_match else 'No'}")

## 📚 4. Recursos adicionales

Estos recursos pueden ayudarte a profundizar en la evaluación automática de traducción:

- 📝 [BLEU score explicado de forma sencilla](https://machinelearningmastery.com/calculate-bleu-score-for-text-python/)
- 🎥 [Vídeo breve sobre BLEU (en inglés)](https://www.youtube.com/watch?v=GyZq57HrbOY)
- 📖 Artículo introductorio sobre métricas léxicas:  
  *Papineni et al. (2002). BLEU: a Method for Automatic Evaluation of Machine Translation.*
- 💻 Documentación de `deep_translator`:  
  https://pypi.org/project/deep-translator/
- 🧪 Jupyter Notebook interactivo sobre evaluación de traducción (huggingface):  
  https://huggingface.co/metrics/bleu

